In [23]:
#!/usr/bin/env python3
"""Print precision, recall, and F1 for original and generated prompts."""

import json
from pathlib import Path


# Set this to the summary.json file you want to read.
SUMMARY_JSON = (
    "gradients_experiments/20260410_071759_llm_cand_sugg_20260301_030139_node11_region3s_qwen4/summary.json"
)

# Use "dev_prf" for dev scores or "balanced_train_prf" for balanced train scores.
SCORE_KEY = "dev_prf"


def get_scores(prompt_info):
    scores = prompt_info.get(SCORE_KEY)
    if scores is None:
        return None

    required_keys = {"precision", "recall", "f1"}
    if not required_keys.issubset(scores):
        missing = ", ".join(sorted(required_keys - set(scores)))
        raise KeyError(f"Missing {missing} in {SCORE_KEY}")

    return scores


def print_prf(scores):
    print(
        f"{scores['precision']:.2f}\t"
        f"{scores['recall']:.2f}\t"
        f"{scores['f1']:.2f}\t"
    )


repo_root = "../"
summary_path = Path(SUMMARY_JSON)
if not summary_path.is_absolute():
    summary_path = repo_root / summary_path

with summary_path.open("r", encoding="utf-8") as f:
    summary = json.load(f)

print_prf(summary["original_prompt"][SCORE_KEY])

for prompt_info in summary.get("generated_prompts", []):
    scores = get_scores(prompt_info)
    if scores is None:
        continue
    print_prf(scores)



41.06	45.03	42.90	
39.74	45.73	42.52	
38.27	46.73	42.08	
40.62	45.73	43.03	
40.54	45.23	42.76	
51.30	39.70	44.76	
30.42	50.75	38.04	
34.24	50.75	40.89	
28.73	52.26	37.08	
29.75	52.76	38.04	
31.91	48.74	38.57	


In [24]:
#!/usr/bin/env python3
"""Print precision, recall, and F1 from pasted agent_evaluate log lines."""

import re


# Paste the evaluate_fn log lines here.
LOG_TEXT = """
[agent_evaluate] evaluate_fn: done in 2904.70s, precision=42.33±4.37, recall=34.39±3.36, f1=37.94±3.78
[agent_evaluate] evaluate_fn: done in 4156.11s, precision=20.07±2.64, recall=54.63±4.83, f1=29.34±3.50
[agent_evaluate] evaluate_fn: done in 3076.31s, precision=44.91±3.08, recall=36.89±2.06, f1=40.46±2.05
[agent_evaluate] evaluate_fn: done in 3447.13s, precision=29.95±4.25, recall=47.19±0.98, f1=36.50±3.50
[agent_evaluate] evaluate_fn: done in 3243.67s, precision=38.46±5.12, recall=42.87±5.25, f1=40.54±5.20
[agent_evaluate] evaluate_fn: done in 3751.95s, precision=35.84±3.51, recall=48.80±5.42, f1=41.32±4.27
[agent_evaluate] evaluate_fn: done in 4260.20s, precision=21.72±2.67, recall=55.68±4.11, f1=31.22±3.36
[agent_evaluate] evaluate_fn: done in 4788.53s, precision=34.30±5.69, recall=56.36±5.94, f1=42.61±6.08
[agent_evaluate] evaluate_fn: done in 2717.36s, precision=12.32±1.41, recall=62.29±5.76, f1=20.57±2.25
"""


METRIC_RE = re.compile(
    r"precision\s*=\s*(?P<precision>[0-9]*\.?[0-9]+)"
    r"(?:\s*(?:±|\+/-|\+-)\s*[0-9]*\.?[0-9]+)?"
    r".*?"
    r"recall\s*=\s*(?P<recall>[0-9]*\.?[0-9]+)"
    r"(?:\s*(?:±|\+/-|\+-)\s*[0-9]*\.?[0-9]+)?"
    r".*?"
    r"f1\s*=\s*(?P<f1>[0-9]*\.?[0-9]+)"
    r"(?:\s*(?:±|\+/-|\+-)\s*[0-9]*\.?[0-9]+)?",
    re.IGNORECASE,
)


for line in LOG_TEXT.splitlines():
    match = METRIC_RE.search(line)
    if not match:
        continue

    precision = float(match.group("precision"))
    recall = float(match.group("recall"))
    f1 = float(match.group("f1"))
    print(f"{precision:.2f}\t{recall:.2f}\t{f1:.2f}\t")


42.33	34.39	37.94	
20.07	54.63	29.34	
44.91	36.89	40.46	
29.95	47.19	36.50	
38.46	42.87	40.54	
35.84	48.80	41.32	
21.72	55.68	31.22	
34.30	56.36	42.61	
12.32	62.29	20.57	
